# Importações e definição

In [54]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
df_vendas = pd.read_csv('../../Datasets/vendas_2023_2024.csv')
user_product = df_vendas[['id_client', 'id_product']]
user_product['comprou'] = 1
print(len(user_product))
user_product = user_product.drop_duplicates()
print(len(user_product))

9895
5413


In [56]:
user_product.head()

,id_client,id_product,comprou
0,42,105,1
1,3,136,1
2,25,139,1
3,20,23,1
4,8,57,1


In [57]:
df_produtos = pd.read_json('../../Datasets/custos_importacao.json')
df_produtos.drop(columns=['historic_data'], inplace=True)
df_produtos.head()

,product_id,product_name,category
0,1,Transponder AIS Maré Magnum,eletrônicos
1,2,Transponder Furuno Marlin,eletrônicos
2,3,Radar Furuno Pulse Leviathan,eletrônicos
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos
4,5,Piloto Automático Furuno Storm,eletrônicos


# Sistema de Recomendação

In [58]:
#Gerando Matrix Usuário × Produto
matrix = (
    user_product
    .pivot_table(
        index='id_client',
        columns='id_product',
        values='comprou',
        aggfunc='max',
        fill_value=0
    )
)
matrix.head()

id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_client,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,0,0,0,0,...,1,1,0,1,1,1,1,1,0,0
2,0,1,1,0,1,1,1,1,1,1,...,1,1,1,1,1,1,0,1,1,1
3,1,1,1,1,1,1,1,0,1,1,...,0,1,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,1,0,0,1,...,1,1,1,1,0,0,1,0,1,1
5,1,1,0,1,1,1,1,1,1,1,...,1,1,1,1,0,1,1,1,1,0


In [59]:
# Gerando matriz de similaridade
# É preciso calcular com a matrix transposta para olhar os itens ao inves dos usuarios
similarity = cosine_similarity(matrix.T)
similarity_df = pd.DataFrame(
    similarity,
    index=matrix.columns,
    columns=matrix.columns
)
similarity_df.head()

id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_product,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.775058,0.737865,0.810191,0.748331,0.769484,0.775058,0.727825,0.711512,0.698771,...,0.795133,0.753819,0.775058,0.769484,0.721605,0.775058,0.872082,0.721605,0.727825,0.775058
2,0.775058,1.000000,0.704295,0.757865,0.714286,0.712931,0.771429,0.750290,0.788811,0.687256,...,0.795192,0.778078,0.771429,0.767772,0.685714,0.742857,0.712931,0.628571,0.750290,0.742857
3,0.737865,0.704295,1.000000,0.800641,0.704295,0.865181,0.732467,0.712396,0.777778,0.707107,...,0.675923,0.739795,0.732467,0.757033,0.704295,0.788811,0.729996,0.732467,0.739795,0.704295
4,0.810191,0.757865,0.800641,1.000000,0.757865,0.753310,0.757865,0.789747,0.773953,0.735980,...,0.831239,0.789747,0.730798,0.779287,0.730798,0.730798,0.805263,0.757865,0.763422,0.757865
5,0.748331,0.714286,0.704295,0.757865,1.000000,0.795192,0.742857,0.694713,0.760639,0.627495,...,0.795192,0.722501,0.714286,0.685511,0.685714,0.714286,0.767772,0.657143,0.722501,0.714286


In [60]:
# Definindo Alvo
item = 'GPS Garmin Vortex Maré Drift'
alvo = df_produtos[df_produtos['product_name'] == item]
alvo = alvo.iloc[0]


In [61]:
# Pegando itens mais similares
print(f"Top 5 similares à {alvo['product_name']} da categoria {alvo['category']}: ")
similares = similarity_df[alvo['product_id']].sort_values(ascending=False).iloc[1:6].reset_index()
similares = (
    similarity_df[alvo['product_id']]
    .sort_values(ascending=False)
    .iloc[1:6]
    .rename('similarity')
    .reset_index()
    .rename(columns={'index': 'id_product'})
    .merge(df_produtos, left_on='id_product', right_on='product_id', how='left')
)
print(similares[['id_product', 'product_name', 'category', 'similarity']].to_string(index=False))

Top 5 similares à GPS Garmin Vortex Maré Drift da categoria eletrônicos: 
 id_product                               product_name    category  similarity
         94           Motor de Popa Volvo Magnum 276HP   propulsão    0.869626
         11        GPS Furuno Swift Leviathan Poseidon eletrônicos    0.868037
         35                         Radar Furuno Swift eletrônicos    0.853913
        115 Cabo de Nylon Delta Force Magnum Leviathan   ancoragem    0.850000
          1                Transponder AIS Maré Magnum eletrônicos    0.850000


# Conclusões

Itens mais similares:
94, Motor de Popa Volvo Magnum 276HP, propulsão, 0.869626
11, GPS Furuno Swift Leviathan Poseidon eletrônicos, 0.868037
A partir dos dados brutos das vendas_2023_2024, o passo a passo para gerar o sistema de 
recomendação foi:
- Pegar somente as colunas de id do cliente e id do produto
- Criar uma coluna para subsituir a quantidade que representa que ocorreu a compra com 1
- Eliminar as duplicatas para não pegar varias vezes o cliente comprando o mesmo item
- Usar pivot_table para usar os ids como indexadores das linhas/colunas
- Preencher os valores existentes com a coluna comprou, e os que não existem com 0
- A partir dessa matriz, calcular as similiaridades entre as colunas
- Isso significa calcular o quão igual é os compradores daquele item com os compradores de outro item
- Para isso cada produto é tratado como um vetor, e cada cliente como uma dimensão
- A similaridade de cosseno é o calculo do angulo entre esses vetores
- Uma limitação desse método é que olhar somente 'comprou' ou não dá menos informação do que,
por exemplo, uma nota para aquele produto
- Além disso, não enxergar a data da compra também pode piorar os resultados de recomendação
para um item para recomendar levar junto pois itens levados em periodos muito distantes
contam o mesmo que itens que foram levados em periodo proximo
- Outros pontos que são possiveis serem melhorados seria um peso associado e normalização
para penalizar clientes que compram muito ou itens muito populares
